# SupplyMind AI — Logistic Regression

Model selection is performed on validation data only.

In [1]:
# -------------------
# Imports
# -------------------

from pathlib import Path

from supplymind.features.predictions.domain.constants import (
    CATEGORICAL_FEATURES,
    NUMERICAL_FEATURES,
)
from supplymind.features.predictions.ml.artifacts import save_model_artifact
from supplymind.features.predictions.ml.evaluation import (
    choose_threshold,
    evaluate_probabilities,
    positive_class_probability,
)
from supplymind.features.predictions.ml.preprocessing import build_preprocessor
from supplymind.features.predictions.ml.reporting import (
    save_evaluation_plots,
    save_feature_importance,
    save_json,
)
from supplymind.features.predictions.ml.training import (
    build_logistic_regression,
    fit_pipeline,
)
from supplymind.features.predictions.ml.workflow import (
    load_clean_syndelay,
    prepare_model_data,
)

In [2]:
# -------------------
# Project configuration
# -------------------

from pathlib import Path

DATASET_PATH = Path("../data/raw/syndelay/syndelay_v1.csv")
REPORT_ROOT = Path("../reports")
MODEL_ROOT = Path("../models")

assert DATASET_PATH.exists(), (
    f"Dataset not found at {DATASET_PATH}. "
    "Place syndelay_v1.csv under data/raw/syndelay/."
)

In [3]:
# -------------------
# Prepare identical model data
# -------------------

df = load_clean_syndelay(DATASET_PATH)
data = prepare_model_data(df)

In [4]:
# -------------------
# Build preprocessing
# -------------------

preprocessor = build_preprocessor(
    NUMERICAL_FEATURES,
    CATEGORICAL_FEATURES,
    scale_numerical=True,
)

In [5]:
# -------------------
# Train model
# -------------------

estimator = build_logistic_regression()
model = fit_pipeline(
    preprocessor,
    estimator,
    data.X_train,
    data.y_train,
)

In [6]:
# -------------------
# Validation probabilities
# -------------------

validation_probability = positive_class_probability(
    model,
    data.X_validation,
)

threshold, threshold_search = choose_threshold(
    data.y_validation,
    validation_probability,
)

print("Selected threshold:", threshold)
threshold_search.sort_values(
    ["f1", "recall"],
    ascending=False,
).head(10)

Selected threshold: 0.22000000000000003


,accuracy,precision,recall,f1,roc_auc,average_precision,true_negative,false_positive,false_negative,true_positive,threshold
2,0.577413,0.577422,0.998886,0.731810,0.740043,0.834546,20,9841,15,13447,0.22
0,0.577198,0.577258,0.999331,0.731798,0.740043,0.834546,9,9852,9,13453,0.20
4,0.577670,0.577637,0.998143,0.731783,0.740043,0.834546,36,9825,25,13437,0.24
5,0.577884,0.577822,0.997474,0.731751,0.740043,0.834546,50,9811,34,13428,0.25
1,0.577198,0.577285,0.999034,0.731739,0.740043,0.834546,13,9848,13,13449,0.21
3,0.577370,0.577430,0.998514,0.731717,0.740043,0.834546,24,9837,20,13442,0.23
6,0.578142,0.578119,0.995840,0.731549,0.740043,0.834546,78,9783,56,13406,0.26
7,0.578313,0.578435,0.993463,0.731159,0.740043,0.834546,114,9747,88,13374,0.27
8,0.578228,0.578739,0.989600,0.730353,0.740043,0.834546,164,9697,140,13322,0.28
9,0.580028,0.580300,0.984252,0.730128,0.740043,0.834546,278,9583,212,13250,0.29


In [9]:
# -------------------
# Validation metrics
# -------------------

metrics = evaluate_probabilities(
    data.y_validation,
    validation_probability,
    threshold=threshold,
)

metrics.to_dict()

{'accuracy': 0.5774128542640312,
 'precision': 0.5774218481621436,
 'recall': 0.9988857524884861,
 'f1': 0.7318095238095238,
 'roc_auc': 0.740042948190666,
 'average_precision': 0.8345463668153176,
 'true_negative': 20,
 'false_positive': 9841,
 'false_negative': 15,
 'true_positive': 13447,
 'threshold': 0.22000000000000003}

In [8]:
# -------------------
# Save candidate reports
# -------------------

MODEL_NAME = "logistic_regression"
REPORT_DIR = REPORT_ROOT / "models" / MODEL_NAME

save_json(
    metrics.to_dict(),
    REPORT_DIR / "validation_metrics.json",
)
threshold_search.to_csv(
    REPORT_DIR / "threshold_search.csv",
    index=False,
)
save_evaluation_plots(
    data.y_validation,
    validation_probability,
    threshold,
    REPORT_DIR,
    "validation",
)
save_feature_importance(
    model,
    REPORT_DIR / "feature_importance",
)

save_model_artifact(
    model,
    {
        "model_name": MODEL_NAME,
        "model_version": "0.1.0-candidate",
        "threshold": threshold,
        "validation_metrics": metrics.to_dict(),
    },
    MODEL_ROOT / "candidates" / MODEL_NAME,
)